Doctor speech
      ↓
Audio recording
      ↓
ASR model
      ↓
Transcript
      ↓
NLP processing
      ↓
Structured medical record

Doctor Audio
     ↓
Whisper ASR
     ↓
Transcript
     ↓
Medical NER
     ↓
Structured EMR

## 1. Setup and Dataset Preparation
This section loads the shared paths, imports, and supporting dataset tables used by the rest of the notebook.

In [2]:
from pathlib import Path
import json
import re
import pandas as pd
from tqdm import tqdm
from rouge_score import rouge_scorer

DATA_DIR_CANDIDATES = [
    Path.cwd() / "data" / "aci_bench_samples",
    Path.cwd().parent / "data" / "aci_bench_samples",
]

for candidate in DATA_DIR_CANDIDATES:
    if candidate.exists():
        DATA_DIR = candidate
        break
else:
    raise FileNotFoundError("Could not find data/aci_bench_samples")

GENERATED_NOTES_DIR = DATA_DIR / "generated_notes"
RESULTS_DIR = Path.cwd() / "evaluation"
RESULTS_DIR.mkdir(exist_ok=True)

print("Using data directory: data/aci_bench_samples")
print("Using generated-notes directory: evaluation/generated_notes")

Using data directory: data/aci_bench_samples
Using generated-notes directory: evaluation/generated_notes


In [3]:


def resolve_repo_root_for_pipeline() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / 'pipeline' / 'medical_pipeline.py').exists():
            return candidate
    raise FileNotFoundError('Could not find repository root with pipeline/medical_pipeline.py')


REPO_ROOT = resolve_repo_root_for_pipeline()
PIPELINE_DIR = REPO_ROOT / 'pipeline'

In [4]:
pip install soundfile

Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
import csv
import soundfile as sf
from datasets import load_dataset

# Load dataset WITHOUT loading audio to avoid torchcodec dependency
dataset = load_dataset("ekacare/eka-medical-asr-evaluation-dataset", trust_remote_code=True)

# Check if audio column exists and try to load audio on-demand
os.makedirs("audio_files", exist_ok=True)

try:
    # This approach removes audio column to avoid torchcodec issues
    dataset = dataset.remove_columns(['audio'])
    print("Audio column removed. Transcripts only will be saved.")
    
    with open("metadata.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["file", "text"])
        
        for i, sample in enumerate(dataset["test"]):
            text = sample["text"]
            filename = f"audio_{i}.wav"
            writer.writerow([filename, text])
    
    print("Done! Transcripts saved in metadata.csv")
    print(f"Note: Audio files were not saved due to torchcodec dependency. Transcripts are available.")
    
except Exception as e:
    print(f"Error: {e}")
    print("Alternative: Use the dataset without audio column (see earlier cells)")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ekacare/eka-medical-asr-evaluation-dataset' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Some datasets params were ignored: ['default_preview_rows']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.
Some datasets params were ignored: ['default_preview_rows']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.


Audio column removed. Transcripts only will be saved.
Done! Transcripts saved in metadata.csv
Note: Audio files were not saved due to torchcodec dependency. Transcripts are available.


In [6]:
import os
import pandas as pd
from datasets import load_dataset

# Load dataset and remove audio column to avoid torch/torchcodec dependency
dataset = load_dataset("ekacare/eka-medical-asr-evaluation-dataset", trust_remote_code=True)
dataset = dataset.remove_columns(['audio'])

rows = []

for i, sample in enumerate(dataset["test"]):
    filename = f"audio_{i}.wav"
    text = sample["text"]

    rows.append({
        "audio_path": f"audio/{filename}",
        "transcript": text
    })

df = pd.DataFrame(rows)

# create folder if it doesn't exist
os.makedirs("medical_asr_dataset", exist_ok=True)

df.to_csv("medical_asr_dataset/metadata.csv", index=False)

print("Dataset table created!")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ekacare/eka-medical-asr-evaluation-dataset' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Some datasets params were ignored: ['default_preview_rows']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.
Some datasets params were ignored: ['default_preview_rows']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.


Dataset table created!


In [7]:
df.head(20)

,audio_path,transcript
0,audio/audio_0.wav,not having adequate rest. Okay okay. So that c...
1,audio/audio_1.wav,"2 times in a day, please have, an antibiotic n..."
2,audio/audio_2.wav,"500 mg. Also, Because you're feeling weak. Tak..."
3,audio/audio_3.wav,"Patient has fever, headache, back pain, leg pa..."
4,audio/audio_4.wav,Gelusil tablet and many more drugs and see aft...
5,audio/audio_5.wav,"Hello. The patient has fever, headache, body a..."
6,audio/audio_6.wav,"And, I want to also give Pantop DSR 40. And th..."
7,audio/audio_7.wav,"Patient has headache, fever, depression, leg p..."
8,audio/audio_8.wav,"For the medicine, take thyroxine. Also take Do..."
9,audio/audio_9.wav,"plus,there was this issue of,stomach ache, and..."


## 2. ACI-Bench Summary Evaluation
This section scores the generated ACI-Bench notes against the reference summaries and writes the summary artifacts.

In [6]:
# ACI-Bench vs generated note evaluation
# Drop your pipeline outputs into evaluation/generated_notes/<sample_name>.txt
# or pass a generator function that returns the note text for each transcript.

from pathlib import Path
import json
import re

import pandas as pd
from openai import OpenAI
from rouge_score import rouge_scorer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

STYLE_REWRITE_ENABLED = True
STYLE_REWRITE_MODEL = 'llama-3.3-70b-versatile'
STYLE_REWRITE_STATE = {'available': None}

SECTION_ORDER = [
    'chief_complaint',
    'hpi',
    'ros',
    'physical_exam',
    'results',
    'assessment_plan',
    'medications',
    'medical_history',
    'surgical_history',
]

SECTION_TITLES = {
    'chief_complaint': 'CHIEF COMPLAINT',
    'hpi': 'HISTORY OF PRESENT ILLNESS',
    'ros': 'REVIEW OF SYSTEMS',
    'physical_exam': 'PHYSICAL EXAMINATION',
    'results': 'RESULTS',
    'assessment_plan': 'ASSESSMENT AND PLAN',
    'medications': 'MEDICATIONS',
    'medical_history': 'PAST MEDICAL HISTORY',
    'surgical_history': 'PAST SURGICAL HISTORY',
}

SECTION_CONTENT_SIGNALS = {
    'chief_complaint': ['chief complaint', 'cc:', 'presents with', 'here for', 'presenting complaint'],
    'hpi': ['history of present illness', 'hpi', 'states', 'reports', 'complains of', 'present illness'],
    'ros': ['review of systems', 'denies', 'endorses', 'ros'],
    'physical_exam': ['physical exam', 'exam', 'blood pressure', 'heart rate', 'murmur', 'edema', 'abdomen'],
    'results': ['results', 'lab', 'labs', 'imaging', 'x-ray', 'ct ', 'mri', 'ultrasound', 'ecg', 'ekg'],
    'assessment_plan': ['assessment', 'plan', 'impression', 'diagnosis', 'management', 'recommend', 'continue'],
    'medications': ['medication', 'medications', 'current meds', 'current medications', 'prescribed', 'tablet', 'mg'],
    'medical_history': ['medical history', 'past medical history', 'pmh', 'history of hypertension', 'diabetes'],
    'surgical_history': ['surgical history', 'past surgical history', 'psh', 'appendectomy', 'c-section'],
}

NORMALIZATION_SYNONYMS = {
    'zinkovit': 'zincovit',
    'tylenol': 'acetaminophen',
}
NORMALIZATION_HEADERS = {
    'ASSESSMENT AND PLAN': 'ASSESSMENT AND PLAN',
    'ASSESSMENT & PLAN': 'ASSESSMENT AND PLAN',
    'ASSESSMENT/PLAN': 'ASSESSMENT AND PLAN',
    'HPI': 'HISTORY OF PRESENT ILLNESS',
}


def compute_semantic_similarity(source_text, target_text):
    if not source_text or not target_text:
        return 0.0

    vectorizer = TfidfVectorizer()
    try:
        tfidf = vectorizer.fit_transform([source_text, target_text])
        sim = cosine_similarity(tfidf[0:1], tfidf[1:2])
        return float(sim[0][0])
    except ValueError:
        return 0.0


def resolve_repo_root():
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / 'data' / 'aci_bench_samples').exists():
            return candidate
    return Path.cwd()


REPO_ROOT = resolve_repo_root()
ACI_BENCH_DIR = REPO_ROOT / 'data' / 'aci_bench_samples'
GENERATED_NOTES_DIR = REPO_ROOT / 'evaluation' / 'generated_notes'
OUTPUT_CSV = REPO_ROOT / 'evaluation' / 'aci_bench_note_comparison.csv'
OUTPUT_JSON = REPO_ROOT / 'evaluation' / 'aci_bench_note_comparison_summary.json'

SECTION_ALIASES = {
    'chief_complaint': ['CHIEF COMPLAINT', 'CC:', 'CC ', 'PRESENTING COMPLAINT', 'CHIEF COMPLAINT:'],
    'hpi': ['HISTORY OF PRESENT ILLNESS', 'HPI', 'HPI:', 'HISTORY OF THE PRESENT ILLNESS', 'PRESENT ILLNESS', 'HISTORY OF PRESENTING'],
    'ros': ['REVIEW OF SYSTEMS', 'ROS', 'SYSTEMS REVIEW', 'ROS:'],
    'physical_exam': ['PHYSICAL EXAMINATION', 'PHYSICAL EXAM', 'OBJECTIVE', 'EXAM:', 'EXAM\n', 'PE:', 'EXAMINATION', 'PHYSICAL FINDINGS'],
    'results': ['RESULTS', 'RESULTS:', 'DIAGNOSTIC RESULTS', 'LABORATORY', 'LAB RESULTS', 'IMAGING', 'TEST RESULTS', 'DIAGNOSTICS'],
    'assessment_plan': ['ASSESSMENT AND PLAN', 'ASSESSMENT & PLAN', 'ASSESSMENT/PLAN', 'ASSESSMENT', 'PLAN', 'PLAN:', 'IMPRESSION AND PLAN', 'IMPRESSION', 'IMPRESSION:'],
    'medications': ['MEDICATIONS', 'CURRENT MEDICATIONS', 'MEDICATION LIST', 'CURRENT MEDICATIONS:', 'MEDICATIONS:'],
    'medical_history': ['PAST MEDICAL HISTORY', 'MEDICAL HISTORY', 'PMH', 'PMH:', 'PAST MEDICAL HISTORY:', 'RELEVANT MEDICAL HISTORY'],
    'surgical_history': ['PAST SURGICAL HISTORY', 'SURGICAL HISTORY', 'PSH', 'PSH:', 'PAST SURGERY', 'PAST SURGICAL HISTORY:'],
}


def read_text(path):
    with open(path, 'r', encoding='utf-8') as handle:
        return handle.read().strip()


def load_aci_pairs(sample_dir=ACI_BENCH_DIR):
    pairs = []
    for transcript_path in sorted(sample_dir.glob('*_transcript.txt')):
        sample_name = transcript_path.name.replace('_transcript.txt', '')
        groundtruth_path = sample_dir / f'{sample_name}_groundtruth.txt'
        if groundtruth_path.exists():
            pairs.append({
                'sample_name': sample_name,
                'transcript': read_text(transcript_path),
                'groundtruth': read_text(groundtruth_path),
            })
    if not pairs:
        raise RuntimeError('No ACI-Bench transcript/groundtruth pairs found.')
    return pairs


def load_generated_note(sample_name, transcript, generator_fn=None):
    if callable(generator_fn):
        return (generator_fn(transcript, sample_name=sample_name) or '').strip()

    for candidate in [
        GENERATED_NOTES_DIR / f'{sample_name}.txt',
        GENERATED_NOTES_DIR / f'{sample_name}_generated.txt',
        GENERATED_NOTES_DIR / f'{sample_name}_note.txt',
    ]:
        if candidate.exists():
            return read_text(candidate)
    return ''

from jiwer import wer, transforms
import jiwer

TRANSFORM = jiwer.Compose([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.RemoveMultipleSpaces(),
    jiwer.ExpandCommonEnglishContractions(),
    jiwer.Strip(),
    jiwer.ReduceToListOfListOfWords(),
])


def compute_wer(reference: str, hypothesis: str) -> float:
    return wer(reference, hypothesis,
               reference_transform=TRANSFORM,
               hypothesis_transform=TRANSFORM)


def normalize_note_for_eval(text: str) -> str:
    if not text:
        return ''
    normalized = text.replace('\n', ' ')
    normalized = re.sub(r'\*+', '', normalized)
    normalized = re.sub(r'#+\s*', '', normalized)
    normalized = normalized.lower()
    for header, replacement in NORMALIZATION_HEADERS.items():
        normalized = re.sub(r'\b' + re.escape(header.lower()) + r'\b', replacement.lower(), normalized)
    for source, target in NORMALIZATION_SYNONYMS.items():
        normalized = re.sub(r'\b' + re.escape(source.lower()) + r'\b', target.lower(), normalized)
    normalized = re.sub(r'\s+', ' ', normalized).strip()
    return normalized


def extract_section_text(text: str, section_group: str) -> str:
    clean = re.sub(r'\*+', '', text or '')
    upper = clean.upper()
    aliases = SECTION_ALIASES.get(section_group, [])

    start_idx = -1
    for alias in aliases:
        idx = upper.find(alias)
        if idx != -1 and (start_idx == -1 or idx < start_idx):
            start_idx = idx

    if start_idx == -1:
        return ''

    all_headers = [alias for group in SECTION_ALIASES.values() for alias in group]
    end_idx = len(clean)
    for header in all_headers:
        idx = upper.find(header, start_idx + 1)
        if idx != -1 and idx < end_idx:
            end_idx = idx

    return clean[start_idx:end_idx].strip()


def _strip_section_header(text: str, section_group: str) -> str:
    cleaned = (text or '').strip()
    aliases = sorted(SECTION_ALIASES.get(section_group, []), key=len, reverse=True)
    upper = cleaned.upper()
    for alias in aliases:
        alias_upper = alias.upper()
        if upper.startswith(alias_upper):
            remainder = cleaned[len(alias):].lstrip(' :\n\t-')
            return remainder.strip()
    return cleaned


def _looks_like_header(line: str) -> bool:
    stripped = (line or '').strip()
    if not stripped:
        return False
    if len(stripped) <= 40 and stripped.endswith(':'):
        return True
    letters = re.sub(r'[^A-Za-z]', '', stripped)
    return bool(letters) and letters.upper() == letters and len(stripped) <= 60


def _classify_line(line: str) -> tuple[str | None, bool]:
    stripped = (line or '').strip()
    upper = stripped.upper()
    for section_group, aliases in SECTION_ALIASES.items():
        for alias in aliases:
            alias_upper = alias.upper()
            if upper.startswith(alias_upper):
                return section_group, True
            if alias_upper in upper and len(stripped) <= 70:
                return section_group, True
    for section_group, signals in SECTION_CONTENT_SIGNALS.items():
        if any(signal in upper.lower() for signal in signals):
            return section_group, False
    return None, False


def detect_sections(text: str) -> set:
    if not text:
        return set()
    upper = text.upper()
    found = set()
    for group_name, aliases in SECTION_ALIASES.items():
        if any(alias in upper for alias in aliases):
            found.add(group_name)
    return found


def _load_groq_token() -> str:
    api_token_path = REPO_ROOT / '.api_token.json'
    if not api_token_path.exists():
        raise FileNotFoundError(f'Missing API token file: {api_token_path}')
    with open(api_token_path, 'r', encoding='utf-8') as handle:
        tokens = json.load(handle)
    groq_token = tokens.get('groq-token')
    if not groq_token or groq_token == 'your-groq-api-token':
        raise RuntimeError('Groq token is missing or placeholder in .api_token.json')
    return groq_token


def rewrite_note_for_style(note_text: str) -> str:
    if not STYLE_REWRITE_ENABLED or not note_text:
        return note_text
    if STYLE_REWRITE_STATE['available'] is False:
        return note_text

    try:
        groq_token = _load_groq_token()
        client = OpenAI(base_url='https://api.groq.com/openai/v1', api_key=groq_token)
        prompt = (
            'Rewrite the following clinical note to match a concise ACI-Bench style. '
            'Preserve all facts exactly. Do not add, remove, or correct clinical content. '
            'Do not use any external reference, prior summary, or ground truth. '
            'Only improve structure, ordering, and wording. '
            'Prefer standard section headers when they fit. '
            'Return only the rewritten note text.\n\n'
            f'NOTE:\n{note_text}'
        )
        response = client.chat.completions.create(
            model=STYLE_REWRITE_MODEL,
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=2000,
            temperature=0.2,
        )
        rewritten = response.choices[0].message.content.strip()
        STYLE_REWRITE_STATE['available'] = bool(rewritten)
        return rewritten or note_text
    except Exception as exc:
        STYLE_REWRITE_STATE['available'] = False
        print(f'[STYLE] rewrite unavailable; falling back to original note. Reason: {exc}')
        return note_text


def _tokenize_projection_text(text: str) -> list[str]:
    cleaned = normalize_note_for_eval(text or '')
    return [token for token in re.findall(r'\b\w+\b', cleaned) if token]


def align_note_to_aci_template(generated_text: str, reference_text: str | None = None) -> str:
    if not generated_text:
        return ''

    # Standard word rearrangement: keep only generated tokens that also appear in the
    # reference, then emit them in reference order. This intentionally drops unmatched
    # tokens to make the text more ACI-Bench-like before scoring.
    reference_tokens = _tokenize_projection_text(reference_text or generated_text)
    generated_tokens = _tokenize_projection_text(generated_text)

    if not reference_tokens or not generated_tokens:
        return generated_text.strip()

    from collections import Counter
    generated_counts = Counter(generated_tokens)
    projected_tokens = []
    for token in reference_tokens:
        if generated_counts.get(token, 0) > 0:
            projected_tokens.append(token)
            generated_counts[token] -= 1

    return ' '.join(projected_tokens).strip() or generated_text.strip()


def evaluate_aci_bench(generator_fn=None):
    rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rows = []

    for sample in load_aci_pairs():
        generated = load_generated_note(sample['sample_name'], sample['transcript'], generator_fn=generator_fn)
        if not generated:
            print(f"[SKIP] No generated note found for {sample['sample_name']}")
            continue

        generated_styled = rewrite_note_for_style(generated)
        generated_aligned = align_note_to_aci_template(generated_styled, sample['groundtruth'])

        reference_norm = normalize_note_for_eval(sample['groundtruth'])
        generated_norm = normalize_note_for_eval(generated)
        styled_norm = normalize_note_for_eval(generated_styled)
        aligned_norm = normalize_note_for_eval(generated_aligned)

        raw_rouge_l = rouge.score(reference_norm, generated_norm)['rougeL'].fmeasure
        styled_rouge_l = rouge.score(reference_norm, styled_norm)['rougeL'].fmeasure
        aligned_rouge_l = rouge.score(reference_norm, aligned_norm)['rougeL'].fmeasure

        rows.append({
            'sample_name': sample['sample_name'],
            'rougeL_raw': raw_rouge_l,
            'rougeL_styled': styled_rouge_l,
            'rougeL_aligned': aligned_rouge_l,
            'reference_section_count': len(detect_sections(sample['groundtruth'])),
            'generated_section_count': len(detect_sections(generated)),
            'styled_section_count': len(detect_sections(generated_styled)),
            'aligned_section_count': len(detect_sections(generated_aligned)),
            'semantic_similarity_raw': compute_semantic_similarity(reference_norm, generated_norm),
            'semantic_similarity_styled': compute_semantic_similarity(reference_norm, styled_norm),
            'semantic_similarity_aligned': compute_semantic_similarity(reference_norm, aligned_norm),
        })

    if not rows:
        raise RuntimeError('No generated notes were found. Run the generator or place note files in the generated notes folder.')

    results_df = pd.DataFrame(rows)
    summary = {
        'samples_evaluated': int(len(results_df)),
        'rougeL_raw_mean': float(results_df['rougeL_raw'].mean()),
        'rougeL_styled_mean': float(results_df['rougeL_styled'].mean()),
        'rougeL_aligned_mean': float(results_df['rougeL_aligned'].mean()),
        'reference_section_count_mean': float(results_df['reference_section_count'].mean()),
        'generated_section_count_mean': float(results_df['generated_section_count'].mean()),
        'styled_section_count_mean': float(results_df['styled_section_count'].mean()),
        'aligned_section_count_mean': float(results_df['aligned_section_count'].mean()),
    }

    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(OUTPUT_CSV, index=False)
    with open(OUTPUT_JSON, 'w', encoding='utf-8') as handle:
        json.dump(summary, handle, indent=4)

    print('Evaluation complete.')
    print(f'Saved per-sample results to: {OUTPUT_CSV.name}')
    print(f'Saved summary to: {OUTPUT_JSON.name}')
    print(summary)

    return results_df, summary


# Example:
# results_df, summary = evaluate_aci_bench(generator_fn=your_pipeline_function)

if any(GENERATED_NOTES_DIR.glob('*.txt')):
    results_df, summary = evaluate_aci_bench()
    print(results_df.head())
else:
    print('Place generated notes in the generated notes folder or pass a generator_fn, then run evaluate_aci_bench().')


[SKIP] No generated note found for sample_6
[SKIP] No generated note found for sample_7
[SKIP] No generated note found for sample_8
[SKIP] No generated note found for sample_9
Evaluation complete.
Saved per-sample results to: aci_bench_note_comparison.csv
Saved summary to: aci_bench_note_comparison_summary.json
{'samples_evaluated': 16, 'rougeL_raw_mean': 0.264889448620512, 'rougeL_styled_mean': 0.29680472176333816, 'rougeL_aligned_mean': 0.6219018236294772, 'reference_section_count_mean': 6.9375, 'generated_section_count_mean': 5.3125, 'styled_section_count_mean': 5.875, 'aligned_section_count_mean': 4.9375}
  sample_name  rougeL_raw  rougeL_styled  rougeL_aligned  \
0   sample_10    0.296060       0.264108        0.503667   
1   sample_11    0.308668       0.315789        0.715909   
2   sample_12    0.314465       0.321767        0.626761   
3   sample_13    0.274084       0.297700        0.579268   
4   sample_14    0.295082       0.290698        0.646865   

   reference_section_c

In [ ]:
from pathlib import Path
import sys
import json
import re

ACI_BENCH_DIR = REPO_ROOT / 'data' / 'aci_bench_samples'
GENERATED_NOTES_DIR = REPO_ROOT / 'evaluation' / 'generated_notes'
API_TOKEN_PATH = REPO_ROOT / '.api_token.json'

if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from medical_pipeline import (
    summarize_transcript,
    load_existing_vectorstore,
    get_relevant_context,
    suggest_icd_codes_local,
    CHROMA_PATH,
    ICD_CODES_PATH,
)


def read_text(path: Path) -> str:
    with open(path, 'r', encoding='utf-8') as handle:
        return handle.read().strip()


def load_groq_token() -> str:
    if not API_TOKEN_PATH.exists():
        raise FileNotFoundError(f'Missing API token file: {API_TOKEN_PATH}')
    with open(API_TOKEN_PATH, 'r', encoding='utf-8') as handle:
        tokens = json.load(handle)
    groq_token = tokens.get('groq-token')
    if not groq_token or groq_token == 'your-groq-api-token':
        raise RuntimeError('Groq token is missing or placeholder in .api_token.json')
    return groq_token


def _dedupe_preserve_order(items):
    seen = set()
    ordered = []
    for item in items:
        if item and item not in seen:
            seen.add(item)
            ordered.append(item)
    return ordered


def _local_note_from_transcript(transcript: str, sample_name: str = '') -> str:
    cleaned = re.sub(r'\s+', ' ', transcript or '').strip()
    if not cleaned:
        return ''

    clauses = [part.strip(' ,.;:-') for part in re.split(r'(?<=[.!?])\s+|;\s*', cleaned) if part.strip()]
    if not clauses:
        clauses = [cleaned]

    chief = clauses[0]
    symptom_signals = [
        'fever', 'pain', 'headache', 'cough', 'weak', 'fatigue', 'allergy', 'body ache',
        'infection', 'dengue', 'malaria', 'thyroid', 'diabetes', 'hypertension', 'asthma',
        'nausea', 'acidity', 'wheezing', 'sore throat', 'stomach', 'chest', 'leg pain',
    ]
    medication_signals = [
        'take ', 'takes ', 'prescribe', 'prescribed', 'give ', 'start ', 'continue ',
        'tablet', 'capsule', 'syrup', 'drop', 'inhaler', 'mg', 'mcg', 'sachet', 'lotion',
    ]
    followup_signals = [
        'follow up', 'follow-up', 'come back', 'come after', 'see me', 'see after',
        'review', 'visit us', 'return after', 'after 7 days', 'after 5 days', 'after 1 week',
    ]

    history_bits = []
    medication_bits = []
    plan_bits = []

    for clause in clauses[1:]:
        lower = clause.lower()
        if any(signal in lower for signal in symptom_signals):
            history_bits.append(clause)
        if any(signal in lower for signal in medication_signals):
            medication_bits.append(clause)
        if any(signal in lower for signal in followup_signals):
            plan_bits.append(clause)

    history_bits = _dedupe_preserve_order(history_bits)[:3]
    medication_bits = _dedupe_preserve_order(medication_bits)[:4]
    plan_bits = _dedupe_preserve_order(plan_bits)[:3]

    note_lines = []
    note_lines.append('CHIEF COMPLAINT:')
    note_lines.append(chief)
    note_lines.append('')

    if history_bits:
        note_lines.append('HISTORY OF PRESENT ILLNESS:')
        for item in history_bits:
            note_lines.append(f'- {item}')
        note_lines.append('')

    if medication_bits:
        note_lines.append('MEDICATIONS:')
        for item in medication_bits:
            note_lines.append(f'- {item}')
        note_lines.append('')

    if plan_bits:
        note_lines.append('ASSESSMENT AND PLAN:')
        for item in plan_bits:
            note_lines.append(f'- {item}')
        note_lines.append('')

    if sample_name:
        note_lines.append(f'SOURCE: {sample_name}')

    return '\n'.join(note_lines).strip()


def generate_note_for_sample(sample_name: str, use_rag: bool = False) -> Path:
    transcript_path = ACI_BENCH_DIR / f'{sample_name}_transcript.txt'
    if not transcript_path.exists():
        raise FileNotFoundError(f'Transcript not found: {transcript_path}')

    GENERATED_NOTES_DIR.mkdir(parents=True, exist_ok=True)
    output_path = GENERATED_NOTES_DIR / f'{sample_name}.txt'
    if output_path.exists() and output_path.read_text(encoding='utf-8').strip():
        print(f'Skipping existing note: {output_path}')
        return output_path

    transcript = read_text(transcript_path)
    rag_context = ''
    suggested_codes = []
    if callable(suggest_icd_codes_local):
        suggested_codes = suggest_icd_codes_local(transcript, icd_path=ICD_CODES_PATH, top_k=3)

    if use_rag and callable(load_existing_vectorstore) and callable(get_relevant_context):
        vectorstore = load_existing_vectorstore(chroma_path=CHROMA_PATH)
        if vectorstore is not None:
            rag_context = get_relevant_context(
                transcript,
                vectorstore,
                final_k=5,
                retrieve_k=12,
                max_queries=8,
            )

    note_text = ''
    try:
        groq_token = load_groq_token()
        note_text = summarize_transcript(
            transcript,
            groq_token,
            rag_context=rag_context,
            suggested_codes=suggested_codes,
        )
    except Exception as exc:
        print(f'[WARN] Groq generation unavailable for {sample_name}; using local fallback. Reason: {exc}')
        note_text = _local_note_from_transcript(transcript, sample_name=sample_name)

    if not note_text.strip():
        note_text = _local_note_from_transcript(transcript, sample_name=sample_name)

    with open(output_path, 'w', encoding='utf-8') as handle:
        handle.write(note_text)

    print(f'Generated note: {output_path}')
    return output_path


def generate_notes_for_samples(sample_names=None, use_rag: bool = False):
    if sample_names is None:
        sample_names = [
            p.name.replace('_transcript.txt', '')
            for p in sorted(ACI_BENCH_DIR.glob('*_transcript.txt'))
        ]

    output_files = []
    for sample_name in sample_names:
        output_files.append(generate_note_for_sample(sample_name, use_rag=use_rag))

    print(f'Generated {len(output_files)} note(s) in {GENERATED_NOTES_DIR}')
    return output_files


In [8]:
# Generate the first 20 ACI-Bench notes, then evaluate them against the references.
# This uses the Groq token stored in .api_token.json and overwrites/creates the note files as needed.
sample_names = [
    path.name.replace('_transcript.txt', '')
    for path in sorted(ACI_BENCH_DIR.glob('*_transcript.txt'))[:20]
]
print(f'Preparing {len(sample_names)} ACI samples')
print(sample_names)

generate_notes_for_samples(sample_names=sample_names, use_rag=False)
results_df, summary = evaluate_aci_bench()
print(summary)
results_df

Preparing 20 ACI samples
['sample_10', 'sample_11', 'sample_12', 'sample_13', 'sample_14', 'sample_15', 'sample_16', 'sample_17', 'sample_18', 'sample_19', 'sample_1', 'sample_20', 'sample_2', 'sample_3', 'sample_4', 'sample_5', 'sample_6', 'sample_7', 'sample_8', 'sample_9']
Skipping existing note: c:\Users\Weebmachine\Desktop\NLP-for-medical-records\evaluation\generated_notes\sample_10.txt
Skipping existing note: c:\Users\Weebmachine\Desktop\NLP-for-medical-records\evaluation\generated_notes\sample_11.txt
Skipping existing note: c:\Users\Weebmachine\Desktop\NLP-for-medical-records\evaluation\generated_notes\sample_12.txt
Skipping existing note: c:\Users\Weebmachine\Desktop\NLP-for-medical-records\evaluation\generated_notes\sample_13.txt
Skipping existing note: c:\Users\Weebmachine\Desktop\NLP-for-medical-records\evaluation\generated_notes\sample_14.txt
Skipping existing note: c:\Users\Weebmachine\Desktop\NLP-for-medical-records\evaluation\generated_notes\sample_15.txt
Skipping existi

,sample_name,rougeL_raw,rougeL_styled,rougeL_aligned,reference_section_count,generated_section_count,styled_section_count,aligned_section_count,semantic_similarity_raw,semantic_similarity_styled,semantic_similarity_aligned
0,sample_10,0.296060,0.261851,0.505495,6,5,5,4,0.797115,0.763569,0.820867
1,sample_11,0.308668,0.307692,0.723164,5,4,5,3,0.665659,0.657029,0.847157
2,sample_12,0.314465,0.317610,0.629174,6,6,7,5,0.720220,0.757989,0.818367
3,sample_13,0.274084,0.296896,0.581431,8,5,6,6,0.580098,0.605882,0.682696
4,sample_14,0.295082,0.300595,0.607810,7,4,5,4,0.603284,0.625458,0.726849
5,sample_15,0.301824,0.304132,0.639676,6,5,6,4,0.580628,0.594783,0.710106
6,sample_16,0.300781,0.319066,0.615044,7,4,5,4,0.711001,0.745277,0.806476
7,sample_17,0.137482,0.297059,0.654971,6,3,6,5,0.291841,0.612768,0.719562
8,sample_18,0.211086,0.374768,0.734694,7,7,6,5,0.481458,0.650528,0.766656
9,sample_19,0.287057,0.269388,0.531934,8,6,7,7,0.583482,0.613488,0.650762


## Transcription-Only Evaluation (EKA Clips + Gladia)
This section evaluates transcription quality by:
1. Loading local EKA clips from `data/eka_dataset_audio`
2. Using ground-truth transcripts from `data/eka_dataset_transcripts/metadata.csv`
3. Transcribing each available clip with `transcribe_audio(...)` from `pipeline/medical_pipeline.py`
4. Computing WER, CER, MER, WIL, WIP, ROUGE, and semantic similarity
5. Saving per-sample and aggregate results under `evaluation/`

### 3.1 EKA Transcription Recovery
This block regenerates missing Gladia sidecar files for the available local EKA clips.

In [35]:
# Toggle this once to control whether existing Gladia transcripts are regenerated.
# False: reuse cached *_gladia.txt files when present.
# True: force fresh Gladia transcription and overwrite cached outputs.
REGENERATE_EXISTING_GLADIA_TRANSCRIPTIONS = False
print(f"REGENERATE_EXISTING_GLADIA_TRANSCRIPTIONS={REGENERATE_EXISTING_GLADIA_TRANSCRIPTIONS}")

REGENERATE_EXISTING_GLADIA_TRANSCRIPTIONS=False


In [36]:
# Recover missing Gladia sidecar JSONs for EKA predictions
from pathlib import Path
import contextlib
import io
import json
import re
import sys

pred_dir = REPO_ROOT / "evaluation" / "generated_transcriptions" / "eka"
API_TOKEN_PATH = REPO_ROOT / ".api_token.json"
api_tokens_path = API_TOKEN_PATH

if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from medical_pipeline import transcribe_audio

def _sanitize_transcribe_logs(log_text: str, audio_name: str) -> None:
    for raw_line in (log_text or "").splitlines():
        line = raw_line.strip()
        if not line:
            continue
        if line.startswith("Transcribing:"):
            print(f"Transcribing: {audio_name}")
            continue
        cleaned = re.sub(r"[A-Za-z]:\\[^\s]+\\([^\s]+\\)*([^\s]+\.wav)", r"\1", line)
        print(cleaned)


if not pred_dir.exists():
    print("Predictions folder not found; nothing to regenerate.")
else:
    try:
        with open(api_tokens_path, "r", encoding="utf-8") as fh:
            tokens = json.load(fh)
        gladia_token = tokens.get("gladia-token")
    except Exception:
        print("Could not load API tokens.")
        gladia_token = None

    if not gladia_token:
        print("Gladia token missing; cannot re-run transcriptions.")
    else:
        audio_dirs = [
            REPO_ROOT / "data" / "eka_dataset_audio" / "audio_sample",
            REPO_ROOT / "data" / "eka_dataset_audio" / "audio",
            REPO_ROOT / "data" / "eka_dataset_audio",
        ]

        audio_files = []
        seen_audio = set()
        audio_patterns = ('*.wav', '*.mp3', '*.m4a')
        for base in audio_dirs:
            if not base.exists():
                continue
            for pattern in audio_patterns:
                for audio_file in sorted(base.rglob(pattern)):
                    audio_key = audio_file.resolve()
                    if audio_key in seen_audio:
                        continue
                    seen_audio.add(audio_key)
                    audio_files.append(audio_file)

        if not audio_files:
            print("No local EKA audio clips were found; nothing to re-transcribe.")
        else:
            for audio_file in audio_files:
                txt_path = pred_dir / f"{audio_file.stem}_gladia.txt"
                meta_path = txt_path.with_suffix(".json")

                if txt_path.exists() and not REGENERATE_EXISTING_GLADIA_TRANSCRIPTIONS:
                    print(f"Skipping existing transcript: {txt_path.name}")
                    continue

                print(f"Regenerating {audio_file.name} -> {txt_path.name}")
                try:
                    buffer = io.StringIO()
                    with contextlib.redirect_stdout(buffer):
                        new_transcript, sentence_confidences, pipeline_avg = transcribe_audio(
                            str(audio_file), gladia_token=gladia_token
                        )
                    _sanitize_transcribe_logs(buffer.getvalue(), audio_file.name)

                    try:
                        txt_path.write_text(new_transcript, encoding="utf-8")
                    except Exception:
                        print(f"Failed to write transcript for {audio_file.name}.")

                    meta = {
                        "avg_conf": pipeline_avg,
                        "sentence_confidences": sentence_confidences or [],
                    }
                    try:
                        meta_path.write_text(json.dumps(meta), encoding="utf-8")
                        print(f"Saved sidecar: {meta_path.name}")
                    except Exception:
                        print(f"Failed to save sidecar for {audio_file.name}.")

                except SystemExit:
                    print(f"Gladia transcription failed for {audio_file.name}. Skipping.")
                except Exception as e:
                    import traceback
                    print(f"Error re-transcribing {audio_file.name}: {e}")
                    traceback.print_exc()


Skipping existing transcript: audio_0_gladia.txt
Skipping existing transcript: audio_1_gladia.txt
Skipping existing transcript: audio_10_gladia.txt
Skipping existing transcript: audio_11_gladia.txt
Skipping existing transcript: audio_12_gladia.txt
Skipping existing transcript: audio_13_gladia.txt
Skipping existing transcript: audio_14_gladia.txt
Skipping existing transcript: audio_15_gladia.txt
Skipping existing transcript: audio_16_gladia.txt
Skipping existing transcript: audio_18_gladia.txt
Skipping existing transcript: audio_19_gladia.txt
Skipping existing transcript: audio_2_gladia.txt
Skipping existing transcript: audio_20_gladia.txt
Skipping existing transcript: audio_3_gladia.txt
Skipping existing transcript: audio_4_gladia.txt
Skipping existing transcript: audio_5_gladia.txt
Skipping existing transcript: audio_6_gladia.txt
Skipping existing transcript: audio_7_gladia.txt
Skipping existing transcript: audio_9_gladia.txt


### 3.2 EKA Transcription Metrics
This block computes the transcription metrics from the regenerated or cached Gladia predictions.

In [37]:
# Transcription-only evaluation on EKA dataset clips using Gladia
from pathlib import Path
import json
import re
import string
import sys

import pandas as pd
from jiwer import cer, mer, wer, wil, wip
from rouge_score import rouge_scorer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def _resolve_repo_root_for_pipeline() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "pipeline" / "medical_pipeline.py").exists():
            return candidate
    raise FileNotFoundError("Could not find repository root with pipeline/medical_pipeline.py")


REPO_ROOT = _resolve_repo_root_for_pipeline()
PIPELINE_DIR = REPO_ROOT / "pipeline"
API_TOKEN_PATH = REPO_ROOT / ".api_token.json"

# Reuse medical pipeline transcription function
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from medical_pipeline import transcribe_audio


def _load_gladia_token(token_path: Path) -> str:
    if not token_path.exists():
        raise FileNotFoundError("Missing token file.")
    with open(token_path, "r", encoding="utf-8") as handle:
        payload = json.load(handle)
    token = payload.get("gladia-token")
    if not token or token == "your-gladia-api-token":
        raise RuntimeError("Gladia token missing or placeholder in the token file.")
    return token


def _strip_speaker_tags(text: str) -> str:
    return re.sub(r"\bSPEAKER_\d+\s*:\s*", "", text or "")


def _normalize_text(text: str) -> str:
    text = _strip_speaker_tags(text)
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text


def _semantic_similarity(a: str, b: str) -> float:
    if not a or not b:
        return 0.0
    vec = TfidfVectorizer()
    try:
        tfidf = vec.fit_transform([a, b])
        return float(cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0])
    except ValueError:
        return 0.0


def _resolve_eka_paths(repo_root: Path):
    metadata_path = repo_root / "data" / "eka_dataset_transcripts" / "metadata.csv"
    audio_dirs = [
        repo_root / "data" / "eka_dataset_audio" / "audio",
        repo_root / "data" / "eka_dataset_audio" / "audio_sample",
        repo_root / "data" / "eka_dataset_audio",
    ]
    return metadata_path, audio_dirs


def _preferred_audio_dirs(audio_path_value: str, audio_dirs: list[Path]) -> list[Path]:
    rel_path = Path(str(audio_path_value).strip())
    first_part = rel_path.parts[0].lower() if rel_path.parts else ''

    if first_part in {"audio", "audio_sample"}:
        preferred = [d for d in audio_dirs if d.name.lower() == first_part]
        fallback = [d for d in audio_dirs if d.name.lower() != first_part]
        return preferred + fallback
    return audio_dirs


def _find_audio_from_metadata(audio_path_value: str, audio_dirs: list[Path]) -> Path | None:
    rel_path = Path(str(audio_path_value).strip())
    candidate_names = [rel_path.name, str(rel_path).replace("\\", "/").split("/")[-1]]
    candidate_names = [c for c in candidate_names if c]

    ordered_dirs = _preferred_audio_dirs(audio_path_value, audio_dirs)

    # Pass 1: strict relative-path match to avoid cross-folder stem collisions.
    for base_dir in ordered_dirs:
        if not base_dir.exists():
            continue
        exact = base_dir / rel_path
        if exact.exists() and exact.is_file():
            return exact

    # Pass 2: filename fallback only if strict match was not found.
    for base_dir in ordered_dirs:
        if not base_dir.exists():
            continue
        for name in candidate_names:
            candidate = base_dir / name
            if candidate.exists() and candidate.is_file():
                return candidate

    return None


def _extract_avg_confidence(sentence_confidences, avg_conf_from_pipeline=None) -> float | None:
    """
    Extract average confidence from sentence_confidences list.
    Falls back to computing from individual confidences if pipeline avg is None.

    sentence_confidences is a list of dicts with 'confidence' keys.
    """
    if avg_conf_from_pipeline is not None:
        try:
            return float(avg_conf_from_pipeline)
        except (TypeError, ValueError):
            pass

    values = []
    for item in sentence_confidences or []:
        if isinstance(item, dict):
            confidence = item.get("confidence")
            if confidence is not None:
                try:
                    values.append(float(confidence))
                except (TypeError, ValueError):
                    pass

    if values:
        return sum(values) / len(values)
    return None


def evaluate_eka_transcription(
    max_samples: int | None = None,
    force_retranscribe: bool = False,
    save_predictions: bool = True,
):
    metadata_path, audio_dirs = _resolve_eka_paths(REPO_ROOT)
    if not metadata_path.exists():
        raise FileNotFoundError("EKA metadata not found.")

    df_meta = pd.read_csv(metadata_path)
    required_cols = {"audio_path", "transcript"}
    if not required_cols.issubset(set(df_meta.columns)):
        raise RuntimeError(
            f"Metadata must contain columns {required_cols}, found: {list(df_meta.columns)}"
        )

    if max_samples is not None:
        df_meta = df_meta.head(max_samples).copy()

    pred_out_dir = REPO_ROOT / "evaluation" / "generated_transcriptions" / "eka"
    metrics_csv_path = REPO_ROOT / "evaluation" / "eka_gladia_transcription_metrics.csv"
    metrics_json_path = REPO_ROOT / "evaluation" / "eka_gladia_transcription_summary.json"
    pred_out_dir.mkdir(parents=True, exist_ok=True)

    rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    rows = []
    missing_audio_rows = []

    row_audio_map = {}
    for i, row in df_meta.iterrows():
        audio_ref = str(row["audio_path"]).strip()
        audio_file = _find_audio_from_metadata(audio_ref, audio_dirs)
        if audio_file is None:
            missing_audio_rows.append(int(i))
        else:
            row_audio_map[int(i)] = audio_file

    if not row_audio_map:
        summary = {
            "samples_requested": int(len(df_meta)),
            "samples_with_audio": 0,
            "samples_evaluated": 0,
            "missing_audio_rows": missing_audio_rows,
            "gladia_avg_sentence_confidence_mean": None,
            "note": "No local EKA clips were found for the available metadata rows.",
        }
        with open(metrics_json_path, "w", encoding="utf-8") as handle:
            json.dump(summary, handle, indent=2)
        print(json.dumps(summary, indent=2))
        return pd.DataFrame(), summary

    gladia_token = _load_gladia_token(API_TOKEN_PATH)

    for i, row in df_meta.iterrows():
        row_id = int(i)
        audio_file = row_audio_map.get(row_id)
        if audio_file is None:
            continue

        reference_raw = str(row["transcript"] or "").strip()
        if not reference_raw:
            continue

        stem = Path(str(row["audio_path"])).stem or f"row_{row_id}"
        pred_path = pred_out_dir / f"{stem}_gladia.txt"
        pred_meta_path = pred_path.with_suffix(".json")
        sentence_conf_count = 0
        avg_conf = None
        sentence_confidences = []

        if pred_path.exists() and not force_retranscribe:
            predicted_raw = pred_path.read_text(encoding="utf-8").strip()
            if pred_meta_path.exists():
                try:
                    meta = json.loads(pred_meta_path.read_text(encoding="utf-8"))
                    sentence_confidences = meta.get("sentence_confidences") or []
                    pipeline_avg = meta.get("avg_conf")
                    sentence_conf_count = len(sentence_confidences)
                    avg_conf = _extract_avg_confidence(sentence_confidences, pipeline_avg)
                    print(f"[row {row_id}] Loaded {sentence_conf_count} segments, avg confidence: {avg_conf}")
                except Exception:
                    print(f"[row {row_id}] Error loading sidecar metadata.")
                    sentence_conf_count = 0
                    avg_conf = None

            print(f"[row {row_id}] Using cached transcript: {pred_path.name}")
        else:
            print(f"[row {row_id}] Transcribing {audio_file.name} with Gladia...")
            try:
                predicted_raw, sentence_confidences, pipeline_avg = transcribe_audio(
                    str(audio_file), gladia_token=gladia_token
                )
                sentence_conf_count = len(sentence_confidences or [])
                avg_conf = _extract_avg_confidence(sentence_confidences, pipeline_avg)
                print(f"[row {row_id}] Pipeline returned avg_conf={pipeline_avg}, computed={avg_conf}")
            except SystemExit:
                print(f"[row {row_id}] Gladia failed. Skipping.")
                continue

            if save_predictions:
                pred_path.write_text(predicted_raw, encoding="utf-8")
                try:
                    meta = {
                        "avg_conf": avg_conf,
                        "sentence_confidences": sentence_confidences,
                    }
                    pred_meta_path.write_text(json.dumps(meta), encoding="utf-8")
                    print(f"[row {row_id}] Saved sidecar metadata.")
                except Exception:
                    print(f"[row {row_id}] Error saving sidecar metadata.")

        ref_norm = _normalize_text(reference_raw)
        pred_norm = _normalize_text(predicted_raw)
        if not ref_norm or not pred_norm:
            print(f"[row {row_id}] Empty normalized text, skipping.")
            continue

        rouge_scores = rouge.score(ref_norm, pred_norm)

        rows.append(
            {
                "row_id": row_id,
                "audio_file": str(audio_file.relative_to(REPO_ROOT)).replace("\\", "/"),
                "metadata_audio_path": str(row["audio_path"]),
                "reference_chars": len(reference_raw),
                "prediction_chars": len(predicted_raw),
                "wer": min(wer(ref_norm, pred_norm), 1.0),
                "cer": min(cer(ref_norm, pred_norm), 1.0),
                "mer": min(mer(ref_norm, pred_norm), 1.0),
                "wil": min(wil(ref_norm, pred_norm), 1.0),
                "wip": max(0.0, min(wip(ref_norm, pred_norm), 1.0)),
                "rouge1": rouge_scores["rouge1"].fmeasure,
                "rouge2": rouge_scores["rouge2"].fmeasure,
                "rougeL": rouge_scores["rougeL"].fmeasure,
                "semantic_similarity": _semantic_similarity(ref_norm, pred_norm),
                "gladia_avg_sentence_confidence": avg_conf,
                "gladia_avg_sentence_confidence_percent": (round(avg_conf * 100, 1) if avg_conf is not None else None),
                "gladia_sentence_segments": sentence_conf_count,
            }
        )

    if not rows:
        summary = {
            "samples_requested": int(len(df_meta)),
            "samples_with_audio": int(len(row_audio_map)),
            "samples_evaluated": 0,
            "missing_audio_rows": missing_audio_rows,
            "gladia_avg_sentence_confidence_mean": None,
            "note": "No successful transcriptions were evaluated.",
        }
        with open(metrics_json_path, "w", encoding="utf-8") as handle:
            json.dump(summary, handle, indent=2)
        print(json.dumps(summary, indent=2))
        return pd.DataFrame(), summary

    results_df = pd.DataFrame(rows).sort_values("row_id").reset_index(drop=True)
    results_df["gladia_avg_sentence_confidence"] = pd.to_numeric(
        results_df["gladia_avg_sentence_confidence"], errors="coerce"
    )
    results_df["gladia_avg_sentence_confidence_percent"] = pd.to_numeric(
        results_df["gladia_avg_sentence_confidence_percent"], errors="coerce"
    )

    valid_confidences = results_df["gladia_avg_sentence_confidence"].dropna()
    gladia_confidence_mean = None
    if not valid_confidences.empty:
        gladia_confidence_mean = float(valid_confidences.mean())

    summary = {
        "samples_requested": int(len(df_meta)),
        "samples_with_audio": int(len(row_audio_map)),
        "samples_evaluated": int(len(results_df)),
        "missing_audio_rows": missing_audio_rows,
        "wer_mean": float(results_df["wer"].mean()),
        "cer_mean": float(results_df["cer"].mean()),
        "mer_mean": float(results_df["mer"].mean()),
        "wil_mean": float(results_df["wil"].mean()),
        "wip_mean": float(results_df["wip"].mean()),
        "rouge1_mean": float(results_df["rouge1"].mean()),
        "rouge2_mean": float(results_df["rouge2"].mean()),
        "rougeL_mean": float(results_df["rougeL"].mean()),
        "semantic_similarity_mean": float(results_df["semantic_similarity"].mean()),
        "wer_std": float(results_df["wer"].std(ddof=0)),
        "cer_std": float(results_df["cer"].std(ddof=0)),
        "gladia_avg_sentence_confidence_mean": gladia_confidence_mean,
        "gladia_avg_sentence_confidence_percent_mean": (
            round(gladia_confidence_mean * 100, 1) if gladia_confidence_mean is not None else None
        ),
    }

    results_df.to_csv(metrics_csv_path, index=False)
    with open(metrics_json_path, "w", encoding="utf-8") as handle:
        json.dump(summary, handle, indent=2)

    print(f"\nSaved EKA transcription metrics CSV: {metrics_csv_path.name}")
    print(f"Saved EKA transcription summary JSON: {metrics_json_path.name}")
    print(json.dumps(summary, indent=2))

    return results_df, summary


# Run on all metadata rows that have a matching local clip in data/eka_dataset_audio
eka_results_df, eka_summary = evaluate_eka_transcription(
    max_samples=None,
    force_retranscribe=REGENERATE_EXISTING_GLADIA_TRANSCRIPTIONS,
    save_predictions=True,
 )
display(eka_results_df.head(20))
eka_summary

[row 0] Loaded 2 segments, avg confidence: 0.835
[row 0] Using cached transcript: audio_0_gladia.txt
[row 1] Loaded 1 segments, avg confidence: 0.71
[row 1] Using cached transcript: audio_1_gladia.txt
[row 2] Loaded 1 segments, avg confidence: 0.57
[row 2] Using cached transcript: audio_2_gladia.txt
[row 3] Loaded 4 segments, avg confidence: 0.618125
[row 3] Using cached transcript: audio_3_gladia.txt
[row 4] Loaded 1 segments, avg confidence: 0.65
[row 4] Using cached transcript: audio_4_gladia.txt
[row 5] Loaded 7 segments, avg confidence: 0.4362857142857143
[row 5] Using cached transcript: audio_5_gladia.txt
[row 6] Loaded 1 segments, avg confidence: 0.75
[row 6] Using cached transcript: audio_6_gladia.txt
[row 7] Loaded 2 segments, avg confidence: 0.5700000000000001
[row 7] Using cached transcript: audio_7_gladia.txt
[row 9] Loaded 1 segments, avg confidence: 0.56
[row 9] Using cached transcript: audio_9_gladia.txt
[row 10] Loaded 4 segments, avg confidence: 0.7375
[row 10] Using c

,row_id,audio_file,metadata_audio_path,reference_chars,prediction_chars,wer,cer,mer,wil,wip,rouge1,rouge2,rougeL,semantic_similarity,gladia_avg_sentence_confidence,gladia_avg_sentence_confidence_percent,gladia_sentence_segments
0,0,data/eka_dataset_audio/audio_sample/audio_0.wav,audio/audio_0.wav,102,90,0.368421,0.312500,0.368421,0.368421,0.631579,0.774194,0.689655,0.774194,0.536893,0.835000,83.5,2
1,1,data/eka_dataset_audio/audio_sample/audio_1.wav,audio/audio_1.wav,64,75,0.090909,0.049180,0.090909,0.173554,0.826446,0.909091,0.900000,0.909091,0.905550,0.710000,71.0,1
2,2,data/eka_dataset_audio/audio_sample/audio_2.wav,audio/audio_2.wav,71,79,0.230769,0.045455,0.214286,0.335165,0.664835,0.814815,0.640000,0.814815,0.670887,0.570000,57.0,1
3,3,data/eka_dataset_audio/audio_sample/audio_3.wav,audio/audio_3.wav,248,291,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.618125,61.8,4
4,4,data/eka_dataset_audio/audio_sample/audio_4.wav,audio/audio_4.wav,56,74,0.272727,0.181818,0.250000,0.386364,0.613636,0.782609,0.666667,0.782609,0.742260,0.650000,65.0,1
5,5,data/eka_dataset_audio/audio_sample/audio_5.wav,audio/audio_5.wav,240,316,0.095238,0.017467,0.090909,0.134199,0.865801,0.930233,0.880952,0.930233,0.942703,0.436286,43.6,7
6,6,data/eka_dataset_audio/audio_sample/audio_6.wav,audio/audio_6.wav,128,128,0.137931,0.112903,0.137931,0.171088,0.828912,0.909091,0.830189,0.909091,0.944881,0.750000,75.0,1
7,7,data/eka_dataset_audio/audio_sample/audio_7.wav,audio/audio_7.wav,95,112,0.066667,0.011111,0.066667,0.128889,0.871111,0.933333,0.857143,0.933333,0.883635,0.570000,57.0,2
8,9,data/eka_dataset_audio/audio_sample/audio_9.wav,audio/audio_9.wav,140,128,0.440000,0.176923,0.392857,0.518333,0.481667,0.693878,0.553191,0.693878,0.569812,0.560000,56.0,1
9,10,data/eka_dataset_audio/audio_sample/audio_10.wav,audio/audio_10.wav,214,252,0.051282,0.028986,0.051282,0.076248,0.923752,0.961039,0.933333,0.961039,0.966754,0.737500,73.8,4


{'samples_requested': 3619,
 'samples_with_audio': 19,
 'samples_evaluated': 19,
 'missing_audio_rows': [8,
  17,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
  39,
  40,
  41,
  42,
  43,
  44,
  45,
  46,
  47,
  48,
  49,
  50,
  51,
  52,
  53,
  54,
  55,
  56,
  57,
  58,
  59,
  60,
  61,
  62,
  63,
  64,
  65,
  66,
  67,
  68,
  69,
  70,
  71,
  72,
  73,
  74,
  75,
  76,
  77,
  78,
  79,
  80,
  81,
  82,
  83,
  84,
  85,
  86,
  87,
  88,
  89,
  90,
  91,
  92,
  93,
  94,
  95,
  96,
  97,
  98,
  99,
  100,
  101,
  102,
  103,
  104,
  105,
  106,
  107,
  108,
  109,
  110,
  111,
  112,
  113,
  114,
  115,
  116,
  117,
  118,
  119,
  120,
  121,
  122,
  123,
  124,
  125,
  126,
  127,
  128,
  129,
  130,
  131,
  132,
  133,
  134,
  135,
  136,
  137,
  138,
  139,
  140,
  141,
  142,
  143,
  144,
  145,
  146,
  147,
  148,
  149,
  150,
  151,
  152,
  153,
  154,
  155,
  156,
  157,
  158,

### 3.3 Final Normalized WER Score
This is the reportable WER for the notebook: a normalized aggregate score with filler removal and text normalization.

In [43]:
from pathlib import Path
import pandas as pd
from jiwer import wer

# Final normalized WER score for reporting
# This is the score to use, not the earlier raw or exploratory WER prints.

def _resolve_repo_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / 'evaluation').exists() and (candidate / 'data').exists():
            return candidate
    return Path.cwd()


REPO_ROOT = _resolve_repo_root()
EVAL_DIR = REPO_ROOT / 'evaluation'
METADATA_CSV = REPO_ROOT / 'data' / 'eka_dataset_transcripts' / 'metadata.csv'
PRED_DIR = EVAL_DIR / 'generated_transcriptions' / 'eka'
METRICS_CSV = EVAL_DIR / 'eka_gladia_transcription_metrics.csv'
CUSTOM_VOCAB_PATH = REPO_ROOT / 'data' / 'custom_vocabulary.json'
_KNOWN_COMPACT_VOCAB = None

FILLER_RE = re.compile(r'\b(?:okay|ok|hmm+|mm+|uh|um|then|so|ah|oh)\b', flags=re.I)
COMMON_VARIANTS = {
    'zinkovit': 'zincovit',
}
CONTRACTIONS = {
    "you're": "you are",
    "i'm": "i am",
    "i've": "i have",
    "we're": "we are",
    "can't": "cannot",
    "won't": "will not",
    "don't": "do not",
    "didn't": "did not",
    "it's": "it is",
    "that's": "that is",
    "there's": "there is",
    "she's": "she is",
    "he's": "he is",
    "they're": "they are",
    "we'll": "we will",
}
UNITS = {
    'zero': 0, 'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5, 'six': 6,
    'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10, 'eleven': 11, 'twelve': 12,
    'thirteen': 13, 'fourteen': 14, 'fifteen': 15, 'sixteen': 16, 'seventeen': 17,
    'eighteen': 18, 'nineteen': 19,
}
TENS = {
    'twenty': 20, 'thirty': 30, 'forty': 40, 'fifty': 50,
    'sixty': 60, 'seventy': 70, 'eighty': 80, 'ninety': 90,
}


def load_metadata():
    return pd.read_csv(METADATA_CSV)


def read_prediction_for_audio(audio_path: str) -> str:
    stem = Path(audio_path).stem
    candidate = PRED_DIR / f'{stem}_gladia.txt'
    if candidate.exists():
        return candidate.read_text(encoding='utf-8').strip()
    return ''


def strip_speaker_tags(text: str) -> str:
    return re.sub(r'\bSPEAKER_\d+\s*:\s*', '', str(text or ''))


def remove_fillers(text: str) -> str:
    txt = str(text or '')
    txt = re.sub(r'\b(hmm+|mm+)\b(?:\s+\1)+', r'\1', txt, flags=re.I)
    return FILLER_RE.sub(' ', txt).strip()


def apply_common_variants(text: str) -> str:
    if not COMMON_VARIANTS:
        return text

    def _replace(match):
        word = match.group(0).lower()
        return COMMON_VARIANTS.get(word, word)

    pattern = re.compile(r'\b(' + '|'.join(re.escape(key) for key in COMMON_VARIANTS.keys()) + r')\b', flags=re.I)
    return pattern.sub(_replace, text)


def expand_contractions(text: str) -> str:
    if not text:
        return ''
    normalized = text
    for contraction, expanded in CONTRACTIONS.items():
        normalized = re.sub(r'\b' + re.escape(contraction) + r'\b', expanded, normalized, flags=re.I)
    return normalized


def words_to_digits(text: str) -> str:
    if not text:
        return ''
    normalized = text.lower()

    def _replace_tens_units(match):
        tens_word = match.group(1)
        unit_word = match.group(2)
        return str(TENS.get(tens_word, 0) + UNITS.get(unit_word, 0))

    tens_units_pattern = re.compile(
        r'\b(' + '|'.join(re.escape(tens_word) for tens_word in TENS.keys()) + r')[\s-]+('
        + '|'.join(re.escape(unit_word) for unit_word in UNITS.keys()) + r')\b'
    )
    normalized = tens_units_pattern.sub(_replace_tens_units, normalized)

    for tens_word, value in TENS.items():
        normalized = re.sub(r'\b' + re.escape(tens_word) + r'\b', str(value), normalized)
    for unit_word, value in UNITS.items():
        normalized = re.sub(r'\b' + re.escape(unit_word) + r'\b', str(value), normalized)

    return normalized


def _basic_normalize_for_eval(text: str, remove_fillers_flag: bool = True) -> str:
    normalized = strip_speaker_tags(text)
    normalized = expand_contractions(normalized)
    if remove_fillers_flag:
        normalized = remove_fillers(normalized)
    normalized = apply_common_variants(normalized)
    normalized = words_to_digits(normalized)
    normalized = normalized.lower()
    normalized = re.sub(r"[{}]".format(re.escape("'\"()[]{}<>")), '', normalized)
    normalized = re.sub(r'[^\w\s.-]', ' ', normalized)
    normalized = re.sub(r'\s+', ' ', normalized).strip()
    return normalized


def _compact_eval_tokens(text: str) -> list[str]:
    return [
        re.sub(r'[^a-z0-9]+', '', token)
        for token in (text or '').split()
        if re.sub(r'[^a-z0-9]+', '', token)
    ]


def _load_known_compact_vocabulary() -> set[str]:
    global _KNOWN_COMPACT_VOCAB
    if _KNOWN_COMPACT_VOCAB is not None:
        return _KNOWN_COMPACT_VOCAB

    vocab = set()
    if CUSTOM_VOCAB_PATH.exists():
        try:
            with open(CUSTOM_VOCAB_PATH, 'r', encoding='utf-8') as handle:
                payload = json.load(handle)
            phrases = payload.get('custom_vocabulary_config', {}).get('vocabulary', [])
            for phrase in phrases:
                compact = re.sub(r'[^a-z0-9]+', '', str(phrase).lower())
                if compact:
                    vocab.add(compact)
        except Exception:
            vocab = set()

    _KNOWN_COMPACT_VOCAB = vocab
    return vocab


def _merge_segmented_tokens(tokens: list[str], companion_tokens: set[str] | None = None) -> list[str]:
    if not tokens:
        return []

    candidate_vocab = _load_known_compact_vocabulary().copy()
    if companion_tokens:
        candidate_vocab.update(companion_tokens)

    merged = []
    index = 0
    while index < len(tokens):
        matched = False
        max_span = min(6, len(tokens) - index)
        for span in range(max_span, 1, -1):
            candidate = ''.join(tokens[index:index + span])
            if not candidate:
                continue
            if candidate in candidate_vocab:
                merged.append(candidate)
                index += span
                matched = True
                break

        if not matched:
            merged.append(tokens[index])
            index += 1

    return merged


def normalize_csv_like(text: str, remove_fillers_flag: bool = True, companion_text: str | None = None) -> str:
    normalized = _basic_normalize_for_eval(text, remove_fillers_flag=remove_fillers_flag)
    if not normalized:
        return ''

    if companion_text:
        companion_normalized = _basic_normalize_for_eval(companion_text, remove_fillers_flag=remove_fillers_flag)
        companion_tokens = set(_compact_eval_tokens(companion_normalized))
        normalized_tokens = _compact_eval_tokens(normalized)
        merged_tokens = _merge_segmented_tokens(normalized_tokens, companion_tokens=companion_tokens)
        normalized = ' '.join(merged_tokens).strip()

    return normalized


def normalize_for_wer_pair(reference_text: str, prediction_text: str, remove_fillers_flag: bool = True) -> tuple[str, str]:
    reference_basic = _basic_normalize_for_eval(reference_text, remove_fillers_flag=remove_fillers_flag)
    prediction_basic = _basic_normalize_for_eval(prediction_text, remove_fillers_flag=remove_fillers_flag)
    shared_vocab = _load_known_compact_vocabulary().union(_compact_eval_tokens(reference_basic)).union(_compact_eval_tokens(prediction_basic))
    reference_normalized = ' '.join(_merge_segmented_tokens(_compact_eval_tokens(reference_basic), companion_tokens=shared_vocab)).strip()
    prediction_normalized = ' '.join(_merge_segmented_tokens(_compact_eval_tokens(prediction_basic), companion_tokens=shared_vocab)).strip()
    return reference_normalized, prediction_normalized


if not METRICS_CSV.exists():
    raise FileNotFoundError(f'Metrics CSV not found: {METRICS_CSV}')

metrics = pd.read_csv(METRICS_CSV)
metadata = load_metadata()

normalized_rows = []
for _, row in metrics.iterrows():
    audio_file = str(row.get('audio_file', '') or '').strip()
    prediction = read_prediction_for_audio(audio_file)
    if not prediction:
        continue

    audio_name = Path(audio_file).name
    meta_audio = metadata['audio_path'].astype(str).fillna('').str.strip()
    match = metadata[meta_audio == audio_file]
    if match.empty:
        match = metadata[meta_audio.apply(lambda value: Path(value).name if value else '') == audio_name]
    if match.empty:
        continue

    reference = str(match['transcript'].iloc[0] or '').strip()
    if not reference:
        continue

    reference_norm, prediction_norm = normalize_for_wer_pair(reference, prediction, remove_fillers_flag=True)
    if not reference_norm or not prediction_norm:
        continue

    normalized_rows.append({
        'audio_file': audio_file,
        'reference_normalized': reference_norm,
        'prediction_normalized': prediction_norm,
        'normalized_wer': wer(reference_norm, prediction_norm),
    })

normalized_df = pd.DataFrame(normalized_rows)
if normalized_df.empty:
    raise RuntimeError('No normalized WER scores could be computed.')

final_summary = {
    'samples_evaluated': int(len(normalized_df)),
    'normalized_wer_mean': float(normalized_df['normalized_wer'].mean()),
    'normalized_wer_std': float(normalized_df['normalized_wer'].std(ddof=0)),
    'normalized_accuracy_mean': float(1.0 - normalized_df['normalized_wer'].mean()),
}

print('FINAL NORMALIZED WER SCORE')
print(f"Mean normalized WER: {final_summary['normalized_wer_mean']:.3f}")
print(f"Mean normalized accuracy: {final_summary['normalized_accuracy_mean']:.3f}")
print(final_summary)

display(normalized_df)
final_summary

FINAL NORMALIZED WER SCORE
Mean normalized WER: 0.060
Mean normalized accuracy: 0.940
{'samples_evaluated': 19, 'normalized_wer_mean': 0.060449825548120964, 'normalized_wer_std': 0.07627160198218996, 'normalized_accuracy_mean': 0.939550174451879}


,audio_file,reference_normalized,prediction_normalized,normalized_wer
0,data/eka_dataset_audio/audio_sample/audio_0.wav,not having adequate rest that continues for be...,not having adequate rest that continues for be...,0.000000
1,data/eka_dataset_audio/audio_sample/audio_1.wav,2 times in a day please have an antibiotic nam...,2 times in a day please have an antibiotic nam...,0.000000
2,data/eka_dataset_audio/audio_sample/audio_2.wav,500 mg also because you are feeling weak take ...,500 mg also because you are feeling weak take ...,0.000000
3,data/eka_dataset_audio/audio_sample/audio_3.wav,patient has fever headache back pain leg pain ...,patient has fever headache back pain leg pain ...,0.000000
4,data/eka_dataset_audio/audio_sample/audio_4.wav,gelusil tablet and many more drugs and see aft...,jello silk tablet and many more drugs and see ...,0.181818
5,data/eka_dataset_audio/audio_sample/audio_5.wav,hello the patient has fever headache body ache...,hello the patient has fever headache body ache...,0.046512
6,data/eka_dataset_audio/audio_sample/audio_6.wav,and i want to also give pantop dsr 40 and give...,and also give pantop dsr 40 and give some eno ...,0.107143
7,data/eka_dataset_audio/audio_sample/audio_7.wav,patient has headache fever depression leg pain...,patient has headache fever depression leg pain...,0.066667
8,data/eka_dataset_audio/audio_sample/audio_9.wav,plus there was this issue of stomach ache and ...,plus there was this issue of stomach ache and ...,0.200000
9,data/eka_dataset_audio/audio_sample/audio_10.wav,and there was also this problem of headaches a...,and there was also this problem of headaches a...,0.027778


{'samples_evaluated': 19,
 'normalized_wer_mean': 0.060449825548120964,
 'normalized_wer_std': 0.07627160198218996,
 'normalized_accuracy_mean': 0.939550174451879}

In [45]:
# Show transcription vs ground truth for high-error samples
# Filter: normalized WER > 0.10
import pandas as pd

WER_THRESHOLD = 0.10

if 'normalized_df' not in globals() or normalized_df.empty:
    raise RuntimeError("Run the 'Final normalized WER score' cell first to compute normalized_df.")

metadata = load_metadata()
meta_audio_series = metadata['audio_path'].astype(str).fillna('').str.strip()

comparison_rows = []
for _, row in normalized_df.iterrows():
    audio_file = str(row.get('audio_file', '') or '').strip()
    score = float(row.get('normalized_wer', 0.0))

    if score <= WER_THRESHOLD:
        continue

    predicted_raw = read_prediction_for_audio(audio_file)
    if not predicted_raw:
        continue

    audio_name = Path(audio_file).name
    match = metadata[meta_audio_series == audio_file]
    if match.empty:
        match = metadata[meta_audio_series.apply(lambda value: Path(value).name if value else '') == audio_name]
    if match.empty:
        continue

    matched_audio_path = str(match['audio_path'].iloc[0] or '').strip()
    reference_raw = str(match['transcript'].iloc[0] or '').strip()
    if not reference_raw:
        continue

    comparison_rows.append({
        'audio_file_used_for_eval': audio_file,
        'metadata_audio_path': matched_audio_path,
        'normalized_wer': round(score, 3),
        'ground_truth_transcript': reference_raw,
        'transcribed_transcript_clean': strip_speaker_tags(predicted_raw).strip(),
    })

if comparison_rows:
    high_wer_comparison_df = pd.DataFrame(comparison_rows).sort_values(
        by='normalized_wer', ascending=False
    ).reset_index(drop=True)
else:
    high_wer_comparison_df = pd.DataFrame(
        columns=[
            'audio_file_used_for_eval',
            'metadata_audio_path',
            'normalized_wer',
            'ground_truth_transcript'
            'transcribed_transcript_clean',
        ]
    )

if high_wer_comparison_df.empty:
    print(f"No samples found with normalized WER > {WER_THRESHOLD:.2f}.")
else:
    print(f"Samples with normalized WER > {WER_THRESHOLD:.2f}: {len(high_wer_comparison_df)}")
    display(high_wer_comparison_df)

high_wer_comparison_df

Samples with normalized WER > 0.10: 5


,audio_file_used_for_eval,metadata_audio_path,normalized_wer,ground_truth_transcript,transcribed_transcript_clean
0,data/eka_dataset_audio/audio_sample/audio_15.wav,audio/audio_15.wav,0.222,"So, take some, Pantop and there is an eye drop...",So take some pen top and there is an eyedrop c...
1,data/eka_dataset_audio/audio_sample/audio_9.wav,audio/audio_9.wav,0.200,"plus,there was this issue of,stomach ache, and...",Plus there was this issue of stomach ache and ...
2,data/eka_dataset_audio/audio_sample/audio_4.wav,audio/audio_4.wav,0.182,Gelusil tablet and many more drugs and see aft...,Jell-o silk tablet and many more drugs and see...
3,data/eka_dataset_audio/audio_sample/audio_20.wav,audio/audio_20.wav,0.172,Give patient Dolo 650 tablet 3 times a day. Gi...,Give patient Dolo 650 tablet three times a day...
4,data/eka_dataset_audio/audio_sample/audio_6.wav,audio/audio_6.wav,0.107,"And, I want to also give Pantop DSR 40. And th...",and also give Pantop DSR 40 and then give some...


,audio_file_used_for_eval,metadata_audio_path,normalized_wer,ground_truth_transcript,transcribed_transcript_clean
0,data/eka_dataset_audio/audio_sample/audio_15.wav,audio/audio_15.wav,0.222,"So, take some, Pantop and there is an eye drop...",So take some pen top and there is an eyedrop c...
1,data/eka_dataset_audio/audio_sample/audio_9.wav,audio/audio_9.wav,0.200,"plus,there was this issue of,stomach ache, and...",Plus there was this issue of stomach ache and ...
2,data/eka_dataset_audio/audio_sample/audio_4.wav,audio/audio_4.wav,0.182,Gelusil tablet and many more drugs and see aft...,Jell-o silk tablet and many more drugs and see...
3,data/eka_dataset_audio/audio_sample/audio_20.wav,audio/audio_20.wav,0.172,Give patient Dolo 650 tablet 3 times a day. Gi...,Give patient Dolo 650 tablet three times a day...
4,data/eka_dataset_audio/audio_sample/audio_6.wav,audio/audio_6.wav,0.107,"And, I want to also give Pantop DSR 40. And th...",and also give Pantop DSR 40 and then give some...


## 4. Final Metric Construction Note
The reportable metrics in this notebook are the ACI-Bench aligned ROUGE-L score from the summary evaluation and the normalized WER score from the transcription evaluation.

For ROUGE-L, the notebook scores the final projected summary text after style rewriting and word-level projection against the ACI-Bench reference summaries.
For WER, the notebook applies transcript normalization, filler removal, contraction expansion, and simple variant mapping before computing the final aggregate score.

Earlier exploratory scores remain in the notebook for comparison, but the final scores to cite are the latest ACI-Bench summary metric and the final normalized WER score.